In [1]:
# ===== Setup and development data =====
import json
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold

from spark_core import *

dev = pd.read_csv("../data/processed/v2_math_dev.csv")
y = dev["risk"]
is_high = (y == "High").astype(int).values

print("Development students:", len(dev), y.value_counts().reindex(RISK_ORDER).to_dict())

Development students: 316 {'High': 104, 'Medium': 132, 'Low': 80}


In [2]:
# ===== Baselines, no learning involved =====
rows = []
for stage, col in [("stage1", "G1"), ("stage2", "G2")]:
    score = -dev[col].values
    rule_pred = dev[col].apply(assign_risk).values
    rows.append({
        "stage": stage,
        "approach": f"Rule, {col} carried forward",
        "avg_precision": average_precision_score(is_high, score),
        "precision_at_90_recall": precision_at_recall(is_high, score),
        **evaluate_predictions(y, rule_pred),
    })

baselines = pd.DataFrame(rows).round(3)
baselines

,stage,approach,avg_precision,precision_at_90_recall,recall_high,precision_high,f1_macro,accuracy,flagged_high,high_predicted_low
0,stage1,"Rule, G1 carried forward",0.810,0.652,0.798,0.728,0.740,0.731,114,0
1,stage2,"Rule, G2 carried forward",0.929,0.845,0.942,0.845,0.888,0.883,116,0


In [3]:
# ===== Tune every model with the same folds and score =====
tuning_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=100)
best_params = {"stage1": {}, "stage2": {}}
tuning_rows = []

for stage, cols in STAGE_COLS.items():
    X = dev[cols]
    for name, grid in TUNING_GRIDS.items():
        search = GridSearchCV(make_pipeline(name), grid, scoring=high_avg_precision,
                              cv=tuning_cv, n_jobs=-1)
        search.fit(X, y)
        best_params[stage][name] = search.best_params_
        tuning_rows.append({"stage": stage, "model": name,
                            "tuning_avg_precision": round(search.best_score_, 3),
                            "best_params": search.best_params_})

tuning_table = pd.DataFrame(tuning_rows)
tuning_table

,stage,model,tuning_avg_precision,best_params
0,stage1,Decision Tree,0.809,"{'clf__max_depth': 5, 'clf__min_samples_leaf': 3}"
1,stage1,Random Forest,0.810,"{'clf__max_depth': 3, 'clf__min_samples_leaf': 1}"
2,stage1,KNN,0.809,"{'clf__n_neighbors': 35, 'clf__weights': 'dist..."
3,stage1,Logistic regression,0.817,{'clf__C': 0.1}
4,stage2,Decision Tree,0.933,"{'clf__max_depth': 6, 'clf__min_samples_leaf':..."
5,stage2,Random Forest,0.948,"{'clf__max_depth': 5, 'clf__min_samples_leaf': 1}"
6,stage2,KNN,0.940,"{'clf__n_neighbors': 35, 'clf__weights': 'unif..."
7,stage2,Logistic regression,0.944,{'clf__C': 3}


In [4]:
# ===== Fresh fold check of the tuned models =====
rows = []
for stage, cols in STAGE_COLS.items():
    X = dev[cols]
    for name in TUNING_GRIDS:
        pipe = make_pipeline(name, **best_params[stage][name])
        runs = oof_high_proba(pipe, X, y, n_repeats=5)
        ap = [average_precision_score(is_high, r) for r in runs]
        par = [precision_at_recall(is_high, r) for r in runs]
        rows.append({"stage": stage, "model": name,
                     "avg_precision": np.mean(ap),
                     "avg_precision_sd": np.std(ap),
                     "precision_at_90_recall": np.mean(par)})

fresh = pd.DataFrame(rows).round(3)
fresh.sort_values(["stage", "avg_precision"], ascending=[True, False])

,stage,model,avg_precision,avg_precision_sd,precision_at_90_recall
3,stage1,Logistic regression,0.823,0.004,0.668
0,stage1,Decision Tree,0.798,0.006,0.659
2,stage1,KNN,0.797,0.007,0.659
1,stage1,Random Forest,0.793,0.005,0.660
5,stage2,Random Forest,0.949,0.003,0.852
7,stage2,Logistic regression,0.946,0.000,0.846
6,stage2,KNN,0.939,0.002,0.834
4,stage2,Decision Tree,0.935,0.009,0.847


In [5]:
# ===== Pre-set rule for the main model =====
proposed = ["Decision Tree", "Random Forest", "KNN"]
stage2 = fresh[(fresh["stage"] == "stage2") & (fresh["model"].isin(proposed))].set_index("model")

best_name = stage2["avg_precision"].idxmax()
best_score = stage2.loc[best_name, "avg_precision"]

if stage2.loc["Random Forest", "avg_precision"] >= best_score - 0.01:
    main_model = "Random Forest"
else:
    main_model = best_name

print(stage2["avg_precision"])
print("Best of the proposed models:", best_name)
print("Main model under the rule:", main_model)

model
Decision Tree    0.935
Random Forest    0.949
KNN              0.939
Name: avg_precision, dtype: float64
Best of the proposed models: Random Forest
Main model under the rule: Random Forest


In [7]:
# ===== Save results for later notebooks =====
baselines.to_csv("../outputs/tables/v2_baselines.csv", index=False)
tuning_table.to_csv("../outputs/tables/v2_tuning.csv", index=False)
fresh.to_csv("../outputs/tables/v2_fresh_folds.csv", index=False)

with open("../outputs/tables/v2_best_params.json", "w") as f:
    json.dump({"best_params": best_params, "main_model": main_model}, f, indent=2)

In [8]:
# ===== Paired bootstrap, tuned models against the rule =====
with open("../outputs/tables/v2_best_params.json") as f:
    best_params = json.load(f)["best_params"]

rng = np.random.default_rng(RANDOM_STATE)

def bootstrap_gap(score_a, score_b, n_boot=1000):
    gaps = []
    n = len(is_high)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if is_high[idx].sum() < 5:
            continue
        gaps.append(average_precision_score(is_high[idx], score_a[idx])
                    - average_precision_score(is_high[idx], score_b[idx]))
    return np.percentile(gaps, [2.5, 50, 97.5]).round(3)

rows = []
for stage, col in [("stage1", "G1"), ("stage2", "G2")]:
    X = dev[STAGE_COLS[stage]]
    rule_score = -dev[col].values
    for name in ["Random Forest", "Logistic regression"]:
        pipe = make_pipeline(name, **best_params[stage][name])
        model_score = np.mean(oof_high_proba(pipe, X, y, n_repeats=5), axis=0)
        low, mid, high = bootstrap_gap(model_score, rule_score)
        rows.append({"stage": stage, "model": name, "median_gap": mid,
                     "ci_low": low, "ci_high": high})

pd.DataFrame(rows)

,stage,model,median_gap,ci_low,ci_high
0,stage1,Random Forest,-0.025,-0.054,-0.000
1,stage1,Logistic regression,0.008,-0.006,0.023
2,stage2,Random Forest,0.021,0.006,0.037
3,stage2,Logistic regression,0.016,0.008,0.027
